# E0 + E1 - verify the rendering fix, and price the image resolution

Follows the P4a smoke run. **No training happens here**: it reloads the LoRA adapter P4a
already saved to Drive, so the whole session is install, load, measure. `Run All` works
top to bottom.

Two questions:

- **E0** - P4a's model produced *correct content in the wrong shape*: every wall it named
  matched ground truth, but it arrived as prose inside a thinking block instead of as
  JSON. The cause was that training and inference rendered the prompt differently. Does
  fixing that make the same adapter emit clean JSON?
- **E1** - P4a measured 7.54 s/step, about 0.94 s per sample, which is slow for a 4B model
  on an L4. How much of that is image resolution?

Both are cheap. E0 decides whether the design is sound before spending hours on a real
run; E1 multiplies through every run that follows.

Expect roughly 10 minutes, most of it the install.

## 1. Install (several minutes)

Commands copied verbatim from the official Unsloth `Qwen3_5_(4B)_Vision.ipynb`. Upstream's `%%capture` is dropped so a failed install is visible rather than resurfacing later as a confusing import error.

In [1]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 147.9 MB/s eta 0:00:0000:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 92ms                                          
Prepared 1 package in 44ms                                               
Uninstalled 1 package in 3ms
Installed 1 package in 9ms                                  
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Resolved 28 packages in 116ms                                        
Prepared 2 packages in 498ms                                             
Uninstalled 1 package in 84ms
Installed 2 packages in 269ms                               
 - transformers==5.5.0
 + transformers==5.2.0
 + typer-slim==0.24.0
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 409ms                                        
Prepared 4 packages in 19.23s                                            
Installed 4 packages in 272ms                               
 + causal-co

## 2. Refuse the wrong runtime

In [2]:
# Refuse the wrong runtime here rather than discover it later.
import torch

capability = torch.cuda.get_device_capability(0)
print("device       ", torch.cuda.get_device_name(0))
print("capability   ", capability)
print("VRAM GiB     ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("bf16 NATIVE  ", torch.cuda.is_bf16_supported(including_emulation=False))

# `is_bf16_supported()` defaults to including_emulation=True and answers True on a Turing
# T4, which has no bf16 hardware at all. Always ask for the native answer.
assert capability >= (8, 0), (
    f"Got capability {capability}; this needs Ampere or newer (L4 is 8.9). "
    "Runtime -> Disconnect and delete runtime, then reconnect having picked L4."
)

device        NVIDIA L4
capability    (8, 9)
VRAM GiB      22.03
bf16 NATIVE   True


## 3. Data and adapter from Drive

In [4]:
import hashlib
import json
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/colab_finetune")
ARCHIVE = "zip_vl_6x6_smoke120_20260822.tar"
DATASET = "smoke_6x6"
EXPECTED_SHA256 = "4424ecc88907173d57b6a7569f68bb259d8a4f4f86da3d1ef523cf0cd10df266"
ADAPTER = DRIVE_DIR / "qwen35_4b_zip_smoke_lora"

drive.mount("/content/drive")

source = DRIVE_DIR / ARCHIVE
digest = hashlib.sha256(source.read_bytes()).hexdigest()
assert digest == EXPECTED_SHA256, f"archive digest {digest} != {EXPECTED_SHA256}"
assert ADAPTER.is_dir(), f"no adapter at {ADAPTER} - was P4a's save cell run?"
print("archive digest OK")
print("adapter found:", ADAPTER)

# Copy onto the VM's own disk first: reading many small files through the Drive FUSE
# mount throttles everything downstream.
!cp "{source}" /content/
!tar -xf /content/{ARCHIVE} -C /content/
!ls /content/{DATASET}/images | wc -l

Mounted at /content/drive
archive digest OK
adapter found: /content/drive/MyDrive/colab_finetune/qwen35_4b_zip_smoke_lora
120


In [5]:
DATA_DIR = Path(f"/content/{DATASET}")

# Verbatim from src/core/vl_models/prompt_variants.FINETUNE_INSTRUCTION - the string the
# adapter was trained with. Querying a fine-tuned checkpoint with anything else asks it a
# question it never saw.
INSTRUCTION = 'Read this Zip puzzle screenshot and reply with ONLY a JSON object.\n"layout" is a 2D array of two-character strings: "  " for an empty cell, "xx" for a blocked cell, and a zero-padded number such as "01" for a waypoint.\n"walls" is a list of {"cell1": [row, col], "cell2": [row, col]} objects, one per thick black bar drawn on a grid line between two neighbouring cells. Report every wall you can see and do not invent any.'

records = [
    json.loads(line)
    for line in (DATA_DIR / "metadata.jsonl").read_text("utf-8").splitlines()
]
# The same split P4a used, so these 4 are genuinely unseen by the adapter.
holdout, train_records = records[:4], records[4:]
print(f"{len(records)} records; holdout {len(holdout)}, seen in training {len(train_records)}")

120 records; holdout 4, seen in training 116


## 4. Load the adapter P4a trained

No training in this notebook.

In [6]:
from unsloth import FastVisionModel

# Loads the adapter P4a saved, not the base model - no training happens in this notebook.
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = str(ADAPTER),
    load_in_4bit = False,
)
FastVisionModel.for_inference(model)
print("adapter loaded")

/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.12/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

adapter loaded


## E0 - does the rendering fix work?

P4a exposed **two** mismatches between how a sample was rendered for training and how the
prompt was rendered for inference:

| | training | inference (as first written) |
|---|---|---|
| thinking | no `<think>` at all - the answer follows `assistant\n` directly | `<think>\n`, opened and never closed |
| content order | `[text, image]` | `[image, text]` |

Either alone puts the model somewhere it was never trained; together they explain the P4a
result exactly.

`build_inference_prompt` fixes both *by construction*: it renders a conversation through
the **same** template call training uses and cuts at the answer, so the prefix matches by
definition rather than by keeping two argument lists in sync. `enable_thinking=False`
would not have been enough - it emits `<think>\n\n</think>\n\n`, which training never
saw either.

⚠ These 4 samples are held out, but they are still *synthetic*. This measures "did it
learn our renderer", not "can it read a real screenshot". The second question needs P3.

In [7]:
import re


def build_inference_prompt(tokenizer, instruction):
    """Render through the SAME template call training uses, then cut at the answer."""
    sentinel = "@@ANSWER@@"
    conv = [
        {"role": "user", "content": [{"type": "text", "text": instruction},
                                     {"type": "image"}]},
        {"role": "assistant", "content": [{"type": "text", "text": sentinel}]},
    ]
    return tokenizer.apply_chat_template(conv, tokenize=False).split(sentinel)[0]


def predict(record, prompt, max_new_tokens=600):
    from PIL import Image

    image = Image.open(DATA_DIR / record["file_name"]).convert("RGB")
    inputs = tokenizer(image, prompt, add_special_tokens=False,
                       return_tensors="pt").to("cuda")
    generated = model.generate(**inputs, max_new_tokens=max_new_tokens,
                               use_cache=True, do_sample=False)
    return tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


def wall_set(payload):
    # .get, not []: a model that answers with JSON but omits "walls" is a *result*, not a
    # crash. The broken-prompt control produced exactly that and took the loop down with it.
    return {tuple(sorted((tuple(w["cell1"]), tuple(w["cell2"]))))
            for w in payload.get("walls") or []
            if isinstance(w, dict) and "cell1" in w and "cell2" in w}


def score(prediction, truth):
    predicted, actual = wall_set(prediction), wall_set(truth)
    hits = len(predicted & actual)
    precision = hits / len(predicted) if predicted else (0.0 if actual else 1.0)
    recall = hits / len(actual) if actual else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return prediction.get("layout") == truth["layout"], hits, len(actual), len(predicted), f1


fixed_prompt = build_inference_prompt(tokenizer, INSTRUCTION)
# The prompt P4a actually used, kept as the control so the comparison is measured.
broken_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"},
                                  {"type": "text", "text": INSTRUCTION}]}],
    add_generation_prompt=True, tokenize=False,
)

print("FIXED  prompt ends:", repr(fixed_prompt[-56:]))
print("BROKEN prompt ends:", repr(broken_prompt[-56:]))

for label, prompt in (("FIXED", fixed_prompt), ("BROKEN (P4a control)", broken_prompt)):
    print(f"\n--- {label} ---")
    parsed = 0
    for record in holdout:
        raw = predict(record, prompt)
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if not match:
            print(f"  {record['file_name']}: NO JSON   raw={raw[:90]!r}")
            continue
        try:
            prediction = json.loads(match.group(0))
        except json.JSONDecodeError as error:
            print(f"  {record['file_name']}: BAD JSON  {error}")
            continue
        parsed += 1
        layout_ok, hits, actual, predicted, f1 = score(prediction, json.loads(record["label"]))
        print(f"  {record['file_name']}: layout={'OK' if layout_ok else 'X '} "
              f"walls {hits}/{actual} (predicted {predicted})  F1={f1:.3f}")
    print(f"  JSON parsed: {parsed}/{len(holdout)}")

FIXED  prompt ends: 'nd|><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
BROKEN prompt ends: 'not invent any.<|im_end|>\n<|im_start|>assistant\n<think>\n'

--- FIXED ---
  images/000000.png: layout=OK walls 10/12 (predicted 12)  F1=0.833
  images/000001.png: layout=OK walls 5/5 (predicted 5)  F1=1.000
  images/000002.png: layout=OK walls 4/4 (predicted 4)  F1=1.000
  images/000003.jpg: layout=OK walls 8/8 (predicted 8)  F1=1.000
  JSON parsed: 4/4

--- BROKEN (P4a control) ---
  images/000000.png: BAD JSON  Expecting value: line 1 column 12 (char 11)
  images/000001.png: NO JSON   raw='The user wants me to output a JSON object with two keys: "layout" and "walls".\n"layout" is'
  images/000002.png: NO JSON   raw='The user wants me to extract a Zip puzzle layout from an image and return only a JSON obje'


KeyError: 'walls'

## E1 - how much is image resolution costing?

P4a measured **7.54 s/step** at an effective batch of 8, about 0.94 s per sample - slow for
a 4B model on an L4. The likeliest reason is resolution: the dataset renders at `cell_size`
72-132, giving images of roughly 500-950 px, and a vision encoder bills by patch count.

This counts **tokens per sample** at several sizes rather than timing training runs. Token
count is what drives the compute, it is instant, and it does not train the adapter further.

If halving the longest side roughly halves the tokens, that multiplies through every run
that follows. Walls are thick black bars and may well survive a downscale - but whether
*accuracy* survives is a separate measurement, not this one.

In [8]:
from PIL import Image

probe = holdout[0]
original = Image.open(DATA_DIR / probe["file_name"]).convert("RGB")
prompt = build_inference_prompt(tokenizer, INSTRUCTION)

# P4a's measured figure, used to turn token ratios into a time estimate.
BASELINE_SECONDS_PER_STEP = 7.54

print(f"source image: {original.size}")
print(f"{'longest side':>13}{'size':>13}{'tokens':>9}{'vs base':>9}{'est s/step':>12}")

baseline_tokens = None
for longest_side in (None, 768, 640, 512, 448, 384):
    if longest_side is None:
        image = original
    else:
        scale = longest_side / max(original.size)
        image = original.resize(
            (max(1, round(original.width * scale)), max(1, round(original.height * scale))),
            Image.Resampling.LANCZOS,
        )
    tokens = tokenizer(image, prompt, add_special_tokens=False,
                       return_tensors="pt")["input_ids"].shape[1]
    baseline_tokens = baseline_tokens or tokens
    ratio = tokens / baseline_tokens
    label = "original" if longest_side is None else str(longest_side)
    print(f"{label:>13}{str(image.size):>13}{tokens:>9}{ratio:>8.2f}x"
          f"{BASELINE_SECONDS_PER_STEP * ratio:>11.2f}s")

print()
print("A proxy, not a promise: attention is superlinear in sequence length, so a real run")
print("may gain more than this suggests. Accuracy at each size is a separate question.")

source image: (656, 656)
 longest side         size   tokens  vs base  est s/step
     original   (656, 656)      529    1.00x       7.54s
          768   (768, 768)      705    1.33x      10.05s
          640   (640, 640)      529    1.00x       7.54s
          512   (512, 512)      385    0.73x       5.49s
          448   (448, 448)      325    0.61x       4.63s
          384   (384, 384)      273    0.52x       3.89s

A proxy, not a promise: attention is superlinear in sequence length, so a real run
may gain more than this suggests. Accuracy at each size is a separate question.


## Done - disconnect

Nothing else is queued for this runtime. `Runtime -> Disconnect and delete runtime`.

It bills at the same rate idle as training (measured 2026-08-22: L4 = 1.54 compute
units/hour), so a session left overnight costs about what the entire P4 stage does.

**What survives a disconnect**: the adapter and the dataset on Drive. **What does not**:
the installed packages and the VM's weight cache - which is why a second experiment is
worth running in the *same* session, and why training a fresh variant means calling
`FastVisionModel.from_pretrained` on the **base** model, never continuing the adapter
above.